In [ ]:
# Here we are using the same Skeleton of chatbot and improving it by adding loops and conditions to store the chat history.
from langgraph.graph import StateGraph,START,END
from langchain_openai import ChatOpenAI
from typing import TypedDict, Literal
import operator
from typing import List,Annotated
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

python-dotenv could not parse statement starting at line 5


In [ ]:
load_dotenv()

llm=ChatOpenAI()

In [ ]:
class JokeState(TypedDict):
    topic:str
    joke:str
    explanation:str


In [ ]:
def generate_joke(state:JokeState):
    prompt=f"generate a joke for the topic :{state['topic']}"
    result=llm.invoke(prompt).content
    return {'joke':result}

def generate_explanation(state:JokeState):
    prompt=f"generate a explanation for the joke :{state['joke']}"
    result=llm.invoke(prompt).content
    return {'explanation': result}


{'review': "I don't like this mobile", 'sentiment': 'negative'}


In [ ]:
graph =StateGraph(JokeState)
graph.add_node('generate_joke',generate_joke)
graph.add_node('generate_explanation',generate_explanation)

graph.add_edge(START,'generate_joke')
graph.add_edge('generate_joke','generate_explanation')
graph.add_edge('generate_explanation',END)

#now build the checkpointer object for InMemorySaver class 
checkpointer=InMemorySaver()


workflow=graph.compile(checkpointer=checkpointer)


#Now create a threadId for this conversation
config={'configurable':{'thread_id':'1'}} 
workflow.invoke({'topic':"pizza"},config=config)